# Grokking reproduction — Colab launcher
Modular addition, Power et al. (2022), arXiv:2201.02177. Settings and sources: `docs/SETTINGS.md`.

**Before you start:** Runtime → Change runtime type → GPU.

Layout: the code runs from local disk (`/content/grokking-repro`, fast); all runs, checkpoints and
results are written to Google Drive (`DRIVE_ROOT`) so they survive disconnects. Re-running a
training cell resumes from the latest checkpoint.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/grokking-repro'   # change if you like
import os; os.makedirs(DRIVE_ROOT, exist_ok=True)
RUNS = f'{DRIVE_ROOT}/runs'
RESULTS = f'{DRIVE_ROOT}/results'

## 2. Get the code
**Option A (zip on Drive):** upload `grokking-repro.zip` to `MyDrive/grokking-repro/`.
**Option B (GitHub):** push the repo to your GitHub and set `GIT_URL`.

In [ ]:
GIT_URL = ''   # e.g. 'https://github.com/<you>/grokking-repro.git'
%cd /content
!rm -rf /content/grokking-repro
if GIT_URL:
    !git clone -q {GIT_URL} /content/grokking-repro
else:
    !unzip -q -o {DRIVE_ROOT}/grokking-repro.zip -d /content
%cd /content/grokking-repro
!pip install -q -e ".[test]"
import torch; print(torch.__version__, torch.cuda.is_available())

## 3. Correctness tests (CPU-sized, ~1 min)

In [ ]:
!python -m pytest -q

Optional: cross-check our split against the **official** `make_data()` (needs the official repo and
its small dependencies).

In [ ]:
!git clone -q https://github.com/openai/grok.git /content/openai-grok 2>/dev/null || true
!pip install -q sympy mod blobfile
!GROK_OFFICIAL_PATH=/content/openai-grok python -m pytest -q tests/test_data.py -k official

## 4. Sanity check: memorize a tiny training set (EXPLORATORY)
1% of the data (94 equations), no weight decay, 1,500 steps. Train accuracy should reach 1.0.

In [ ]:
!python scripts/train.py --config exploratory.yaml --variant tiny_overfit --out {RUNS}/explore/tiny_overfit
!python scripts/analyze.py {RUNS}/explore/tiny_overfit --title "EXPLORATORY: tiny overfit" --out {RESULTS}/explore/tiny_overfit

## 5. Reproduction runs (paper Fig. 2 right protocol)
15 training fractions (0.10–0.80, measured from the figure) × 3 seeds × 10^5 steps = 45 runs.
Watch the `elapsed` column of the first run to estimate the total; then widen `RUN_NOW`
across sessions. If Colab disconnects, re-run the cell: finished runs are skipped (`DONE`)
and unfinished ones resume from `checkpoints/latest.pt` on Drive.

In [ ]:
import yaml
rep = yaml.safe_load(open('configs/reproduction.yaml'))
ALL_PCTS = rep['sweep']['train_pcts']                    # 10..80 step 5, from paper Fig. 2 (right)
SEEDS = list(range(rep['sweep']['seeds_per_pct']))       # 3 seeds; values 0,1,2 are OUR choice
# Run order only (not a setting change): the region where the paper's curve changes comes first.
PRIORITY = [10, 15, 20, 25]
TRAIN_PCTS = PRIORITY + [p for p in ALL_PCTS if p not in PRIORITY]
RUN_NOW = TRAIN_PCTS[:4]       # widen this list over later sessions until all 15 are done

for pct in RUN_NOW:
    for seed in SEEDS:
        out = f'{RUNS}/repro/p{pct:g}_s{seed}'   # f-string is evaluated in Python
        if os.path.exists(f'{out}/DONE'):
            print('done:', out); continue
        !python scripts/train.py --config reproduction.yaml --train-pct {pct} --seed {seed} --out {out}


## 6. Analysis (training + validation only)
Plots accuracy and loss vs. step (log x) for all seeds of each fraction, and writes timing
metrics and the delayed-generalization label defined in `grokrepro/analysis.py`.

In [ ]:
from IPython.display import Image, display
done_pcts = [p for p in TRAIN_PCTS if os.path.exists(f'{RUNS}/repro/p{p:g}_s0/metrics.jsonl')]
for pct in done_pcts:
    tag = f'p{pct:g}'
    title = f'REPRODUCTION: (a+b) mod 97, {pct:g}% train'
    !python scripts/analyze.py {RUNS}/repro/{tag}_s* --title "{title}" --out {RESULTS}/repro/{tag}
    display(Image(f'{RESULTS}/repro/{tag}.png'))

In [ ]:
# Figure-2-style summary: best validation accuracy vs. fraction, with the paper's approximate curve
!python scripts/sweep_summary.py {RUNS}/repro --out {RESULTS}/repro/fig2_addition
display(Image(f'{RESULTS}/repro/fig2_addition.png'))

## 7. Final test evaluation (run once, after all reproduction runs are finished)

In [ ]:
!python scripts/final_eval.py {RUNS}/repro/*

In [ ]:
drive.flush_and_unmount()   # make sure everything is written to Drive before closing